Today's topics:
* arithmetic, unit conversion, and formatted output
* manipulating vectors with `numpy`

A little units humor from XKCD

<img src="https://imgs.xkcd.com/comics/uncanceled_units.png" alt="xkcd comic: one person says a refrigerator uses only 3 kWh per day, and another asks whether it will fit because the kitchen ceiling is only 50 gallons per square foot" height=340/>

"Uncanceled Units", [xkcd 3038](https://xkcd.com/3038/) by Randall Munroe,
[CC BY-NC 2.5](https://xkcd.com/license.html).

A case of mistaken identity

In 1999, NASA lost the Mars Climate Orbiter
because one team sent thrust data in pound-force-seconds and the other expected
newton-seconds. The spacecraft came in too low and burned up in the atmosphere.
That was a \$125 million unit conversion error.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/4/42/Mars_Climate_Orbiter_during_tests.jpg/960px-Mars_Climate_Orbiter_during_tests.jpg" alt="Engineers preparing the Mars Climate Orbiter during acoustic testing" height=300/>

NASA photograph, 1998. [Source](https://commons.wikimedia.org/wiki/File:Mars_Climate_Orbiter_during_tests.jpg), [public domain](https://commons.wikimedia.org/wiki/File:Mars_Climate_Orbiter_during_tests.jpg#Licensing).

Let's see what that error looked like in numbers. The ground team sent an
impulse of 4.45 pound-force-seconds, but the spacecraft expected
newton-seconds:

In [1]:
impulse_lbf_s = 4.45
impulse_N_s = impulse_lbf_s * 4.44822   # 1 lbf·s = 4.44822 N·s
print(impulse_lbf_s)
print(impulse_N_s)

4.45
19.794579000000002


The spacecraft used 4.45 when it should have used 19.79. That
factor-of-four error accumulated over months of navigation corrections and
put the orbiter 170 km too close to Mars.

Here is a quick look at what unit-aware code can do. You do not need to learn
this syntax or use it in your work:

In [2]:
#@title Optional tool glimpse: Pint keeps units attached (click ▶ to run) { display-mode: "form" }
try:
    import pint
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pint==0.25.3"])
    import pint

units = pint.UnitRegistry()

In [3]:
impulse = 4.45 * units.pound_force * units.second
print(impulse)
print(impulse.to(units.newton * units.second))

4.45 force_pound * second
19.79458618790923 newton * second


# Units are everywhere

Python is great at arithmetic but has absolutely no concept of a "gram".
It won't catch a unit error for you.
Specialized software like Pint can, but we need to remember to use it.
So we need to be deliberate about tracking units, and what tools can make that easier.

Materials properties show up in all sorts of unit systems.
Your lab balance reads grams and your graduated cylinder reads milliliters, but the SI density is reported in $\mathrm{kg/m}^3$.
Some engineering references still use imperial units like $\mathrm{lb/ft}^3$.

Let's start with a sample we measured in the chemistry lab:

In [4]:
mass_g = 44.5
volume_mL = 5.0

What should we do with these? Arithmetic!

# Arithmetic operators

Here are the five arithmetic operators you'll use most often:

| operator | meaning | example | result |
|---|---|---|---|
| `+` | addition | `4 + 3` | `7` |
| `-` | subtraction | `4 - 3` | `1` |
| `*` | multiplication | `4 * 3` | `12` |
| `/` | division | `4 / 3` | `1.333...` |
| `**` | exponent | `4 ** 3` | `64` |

Let's use division to compute density:

In [5]:
density_g_per_mL = mass_g / volume_mL
print(density_g_per_mL)

8.9


## Caution about exponents

A common mistake: exponents use `**`, not `^`.
Let's see how this plays out.
To get the volume of a cube, you cube the side length: $V = a^3$.

In [6]:
a = 4
# correct
print(a ** 3)
# incorrect
print(a ^ 3)

64
7


`a ** 3` gives 64, which is the cube we want.
`a ^ 3` gives 7, which is something completely different (bitwise XOR).

Python won't give you an error here. It just silently gives you the wrong quantity.

## Order of operations

Order of operations works the same as in algebra: parentheses first, then exponents, then `*`/`/`, then `+`/`-`.

When in doubt, add parentheses. Let's see how they change the result:

In [7]:
print(mass_g / volume_mL + 1)
print(mass_g / (volume_mL + 1))

9.9
7.416666666666667


Those give very different answers. If a calculation is getting complicated,
it often helps to break it into named steps:

In [8]:
adjusted_volume_mL = volume_mL + 1
adjusted_density = mass_g / adjusted_volume_mL
print(adjusted_density)

7.416666666666667


Put spaces around `+`, `-`, `*`, `/`, `=`. It makes your code easier to read.

## Reading error messages

Errors are normal. You will see a lot of them this semester. **Strategy: read
the last line first.** That's where Python tells you what went wrong.

**`NameError`** means Python doesn't recognize a name. Usually a typo:

In [9]:
# EXPECTED-ERROR: this cell intentionally fails
density = 8.9
print(denisty)

NameError: name 'denisty' is not defined

`name 'denisty' is not defined`. We wrote `denisty` instead of `density`.

**`TypeError`** means you tried an operation on the wrong type:

In [10]:
# EXPECTED-ERROR: this cell intentionally fails
'5' + 5

TypeError: can only concatenate str (not "int") to str

`can only concatenate str (not "int") to str`.
Python tried string concatenation but found an `int`.
Fix with `int('5') + 5`.

You already saw **`SyntaxError`** last class when `mass g = 44.5` failed.
That one means Python can't even read your code.

### [Check your understanding]

Now let's practice. Each code cell below has a bug.
Some crash with an error; one gives the wrong answer silently.
Find and fix each bug.

**Bug 1:** compute the area of a circle with radius 5 cm

In [11]:
# BUG: this cell runs but gives the wrong answer
radius = 5
area = 3.14159 * (radius ^ 2)
print(area)

21.99113


**Bug 2:** convert a temperature from Celsius to Fahrenheit

In [12]:
# EXPECTED-ERROR: this cell has a bug to find and fix
temp_c = 1538
temp_f = temp_c * 9/5 + 32
print(tmp_f)

NameError: name 'tmp_f' is not defined

**Bug 3:** compute and display a lattice parameter

In [13]:
# EXPECTED-ERROR: this cell has a bug to find and fix
a_nm = '0.3615'
a_cm = a_nm * 1e-7
print(a_cm)

TypeError: can't multiply sequence by non-int of type 'float'

## Scientific notation

Materials science uses a lot of very large and very small numbers.
Python has a built-in way to write them using `e` notation:

| Written form | Python | What it is |
|---|---|---|
| $6.022 \times 10^{23}$ | `6.022e23` | Avogadro's number |
| $1.6 \times 10^{-19}$ | `1.6e-19` | electron charge (C) |
| $17 \times 10^{-6}$ | `17e-6` | thermal expansion coeff. ($/^\circ\mathrm{C}$) |

Let's try a few:

In [14]:
print(6.022e23)
print(17e-6)
print(type(17e-6))

6.022e+23
1.7e-05
<class 'float'>


Notice that `17e-6` is a `float`.
Any time you use `e` notation, Python treats it as a decimal number.

## Units and unit conversion

This brings us back to the problem we started with.
In materials science, you'll run into at least three unit systems:

| System | Where you see it | Density of copper |
|---|---|---|
| CGS | lab measurements, older handbooks | 8.96 $\mathrm{g/cm}^3$ |
| SI | simulations, journal papers | 8960 $\mathrm{kg/m}^3$ |
| Imperial | some engineering specs | 559 $\mathrm{lb/ft}^3$ |

Those are all the same physical quantity.
The number just looks very different depending on which units you use.

`8.96` could be density in $\mathrm{g/cm}^3$, or it could be length in meters.
Python has no idea which one you mean.
It checks whether your code is valid Python instructions, but nothing beyond that.

Let's convert copper's density from $\mathrm{g/cm}^3$ to $\mathrm{kg/m}^3$:

In [15]:
density_g_per_cm3 = 8.96

# 1 g/cm^3 = 1000 kg/m^3
density_kg_per_m3 = density_g_per_cm3 * 1000
print(density_kg_per_m3)

8960.0


The best way to keep track is with descriptive variable names like `density_g_per_cm3` and comments that explain your reasoning.

Another (humorous) word of caution on units:

<img src="https://imgs.xkcd.com/comics/dimensional_analysis.png" alt="xkcd comic: a teacher presents an equation combining Planck energy, the pressure at the Earth's core, Prius gas mileage, and the width of the English Channel, which equals pi and has consistent units" height=380/>

"Dimensional Analysis", [xkcd 687](https://xkcd.com/687/) by Randall Munroe,
[CC BY-NC 2.5](https://xkcd.com/license.html).

### [Check your understanding]

An aluminum-alloy sample has mass 39.75 g and volume 15.0 mL.
Pure aluminum has a handbook density of 2.70 g/mL.

In the code cell below:

1. Define variables for mass and volume
2. Compute the density in g/mL
3. Convert to $\mathrm{kg/m}^3$
4. Print both values

*Optional challenge*

5. Compute the signed percent deviation from pure aluminum:

$$\text{percent deviation} = \frac{\rho_{\text{sample}} - \rho_{\text{reference}}}{\rho_{\text{reference}}} \times 100 \%$$

6. Print the percent deviation

# `print()` and f-strings

So far we've been printing raw numbers.
In a technical document you would never write just "8.9" with no label or units.
Let's fix that.

First, an important note about how Colab displays things.
The last line of a cell auto-displays:

In [16]:
density_g_per_mL

8.9

But if it's not the last line, you need `print()`:

In [17]:
mass_g = 44.5
volume_mL = 5.0
density_g_per_mL   # does NOT display, not the last line
print(mass_g)      # displays, it explicitly calls print()
volume_mL

44.5


5.0

Now let's add labels. **f-strings** let you embed variables directly in text.
Put `f` before the opening quote, and wrap variable names in `{}`:

In [18]:
print(f"The density is {density_g_per_mL} g/mL")

The density is 8.9 g/mL


In [19]:
print(f"A sample with mass {mass_g} g and volume {volume_mL} mL has density {mass_g / volume_mL} g/mL")

A sample with mass 44.5 g and volume 5.0 mL has density 8.9 g/mL


## Formatting numbers

That's a lot of decimal places.
To round, add `:.2f` inside the braces.
The `2` means two decimal places, and `f` means fixed-point:

In [20]:
density_g_per_mL = mass_g / volume_mL
print(f"The density is {density_g_per_mL:.2f} g/mL")

The density is 8.90 g/mL


For very small or very large numbers, `:.2e` gives scientific notation:

In [21]:
lattice_cm = 3.615e-8
print(f"Lattice parameter: {lattice_cm:.2e} cm")

Lattice parameter: 3.62e-08 cm


### [Check your understanding]

The volume of a sphere is $V = \frac{4}{3}\pi r^3$.

In the code cell below:

1. Define a variable `r_cm` with value `3` (radius in cm) and `pi` with value `3.14159`
2. Compute the volume in $\mathrm{cm}^3$
3. Convert to $\mathrm{m}^3$ (1 $\mathrm{cm}^3$ = $10^{-6}$ $\mathrm{m}^3$, so multiply by `1e-6`)
4. Print both values as labeled f-strings, with
- $\mathrm{cm}^3$ value rounded to 2 decimal places and
- $\mathrm{m}^3$ value in scientific-notation style (use `:.2e` as the format specifier)

# representing vectors with `numpy` arrays

Everything we've done so far has been with single numbers.
But in the real world, you rarely measure just one sample.
What if you have ten? One hundred? One million?

It turns out there's a library called NumPy that adds support for exactly this:
arrays, which are variables that hold many numbers at once.
`import` loads the library, and `as np` gives it a short nickname (alias) so we don't have to type `numpy` every time:

In [22]:
import numpy as np

# synthetic teaching data -- five metal samples
mass = np.array([27.0, 41.6, 13.5, 54.2, 20.8])
volume = np.array([10.0, 15.0, 5.0, 20.0, 8.0])

print(mass)
print(volume)

[27.  41.6 13.5 54.2 20.8]
[10. 15.  5. 20.  8.]


`np.array([])` creates an array, which is one variable that holds many numbers in order.
The same operators we just learned work on arrays, applied entry by entry ("elementwise"):

In [23]:
density = mass / volume
print(density)

[2.7        2.77333333 2.7        2.71       2.6       ]


That one line computed all five densities.
First mass with first volume, second with second, and so on.
This is where the code starts to pay off compared to a calculator.
If you had 100 samples, or 1,000,000, the code would look exactly the same.

In [24]:
print(f"Densities: {density}")
print(f"Lightest: {np.min(density):.2f} g/mL")
print(f"Heaviest: {np.max(density):.2f} g/mL")

Densities: [2.7        2.77333333 2.7        2.71       2.6       ]
Lightest: 2.60 g/mL
Heaviest: 2.77 g/mL


## Operators depend on type

This connects back to something we saw last class with strings.
The same `+` operator does different things depending on what types you give it:

In [25]:
print(5 + 3)
print('5' + '3')
print(np.array([5, 3]) + np.array([10, 20]))

8
53
[15 23]


This is important to keep in mind: these symbols are not arithmetic, they are Python operators!
Their behavior will be highly contextual and you need to understand what you're asking the computer to do.

## Further reading

- [w3schools: Python operators](https://www.w3schools.com/python/python_operators.asp)
- [w3schools: Python string formatting](https://www.w3schools.com/python/python_string_formatting.asp)
- [VanderPlas: The basics of NumPy arrays](https://jakevdp.github.io/PythonDataScienceHandbook/02.02-the-basics-of-numpy-arrays.html)